In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/notebooks/puttadatta/fintech-project/unified_dev_sample.parquet
/kaggle/input/notebooks/puttadatta/fintech-project/__results__.html
/kaggle/input/notebooks/puttadatta/fintech-project/unified_user_exposure.parquet
/kaggle/input/notebooks/puttadatta/fintech-project/__notebook__.ipynb
/kaggle/input/notebooks/puttadatta/fintech-project/__output__.json
/kaggle/input/notebooks/puttadatta/fintech-project/custom.css


In [2]:
import pandas as pd

In [3]:
dev_df = pd.read_parquet("/kaggle/input/notebooks/puttadatta/fintech-project/unified_dev_sample.parquet")
dev_df.head()

,user_id,step,amount,balance_after,isFraud,role,rolling_peak_3
0,C1000001725,46,-8424.740234,0.0,0,origin,0.0
1,C1000018372,156,-108365.468750,18449576.0,0,origin,0.0
2,C1000022742,18,-2368.679932,0.0,0,origin,0.0
3,C1000069522,329,-590353.812500,13756703.0,0,origin,0.0
4,C1000089317,140,-283090.375000,0.0,0,origin,0.0


In [4]:
dev_df.shape

(418269, 7)

In [5]:
dev_df.describe()

,step,amount,balance_after,isFraud,rolling_peak_3
count,418269.000000,4.182690e+05,4.182690e+05,418269.000000,418269.000000
mean,243.768118,-1.056744e+03,1.036228e+06,0.001308,0.028824
std,142.285046,5.957035e+05,3.114764e+06,0.036140,0.129388
min,1.000000,-5.894475e+07,0.000000e+00,0.000000,0.000000
25%,156.000000,-7.604440e+04,0.000000e+00,0.000000,0.000000
50%,240.000000,-3.064300e+02,1.166264e+04,0.000000,0.000000
75%,335.000000,7.228514e+04,5.507579e+05,0.000000,0.000000
max,743.000000,5.957950e+07,2.059364e+08,1.000000,1.000000


In [6]:
dev_df['rolling_peak_3'].quantile([0.90,.95,.99])

0.90    0.000000
0.95    0.165545
0.99    0.828843
Name: rolling_peak_3, dtype: float64

In [7]:
dev_df = dev_df.rename(columns={'rolling_peak_3':'drawdown_ratio'})
dev_df.columns

Index(['user_id', 'step', 'amount', 'balance_after', 'isFraud', 'role',
       'drawdown_ratio'],
      dtype='object')

In [8]:
per_drawdown = 100*(dev_df['drawdown_ratio']>.2).sum()/len(dev_df)
print("Transactions that have more than 20% drawdown are:",per_drawdown)

Transactions that have more than 20% drawdown are: 4.525795600438952


In [9]:
high_dd = dev_df['drawdown_ratio']>.2
fraud_rate_in20 = dev_df.loc[high_dd,'isFraud'].mean()
low_dd = dev_df['drawdown_ratio']<=.2
fraud_rate_out20 = dev_df.loc[low_dd,'isFraud'].mean()
print("fraud rate below 20% of drawdown ratio:",fraud_rate_in20)
print("fraud rate above 20% of drawdown ratio:",fraud_rate_out20)

fraud rate below 20% of drawdown ratio: 0.0
fraud rate above 20% of drawdown ratio: 0.001369763534240332


In [10]:
dev_df['high_drawdown_flag'] = (dev_df['drawdown_ratio']>.20)

In [11]:
step_risk_table = (
    dev_df
    .groupby('step')
    .agg(
        transaction_count = ('user_id','count'),
        drawdown_rate = ('high_drawdown_flag','mean'),
        fraud_rate = ('isFraud','mean'),
        avg_drawdown = ('drawdown_ratio','mean')
    )
)

step_risk_table.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown
step,,,,
1,180,0.011111,0.0,0.010098
2,61,0.049180,0.0,0.029789
3,31,0.064516,0.0,0.030009
4,28,0.071429,0.0,0.037322
5,46,0.130435,0.0,0.063839


In [12]:
step_risk_table_2 = step_risk_table
step_risk_table_3 = step_risk_table

In [13]:
step_risk_table_2.sort_values(
    by = ['drawdown_rate'],
    ascending = False,
    inplace = True
)
step_risk_table_2.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown
step,,,,
445,4,0.250000,0.000000,0.242620
483,16,0.187500,0.000000,0.127306
630,11,0.181818,0.000000,0.087587
338,23,0.173913,0.000000,0.112854
531,19,0.157895,0.052632,0.135037


In [14]:
step_risk_table_3.sort_values(
    by = ['fraud_rate'],
    ascending = False
)
step_risk_table_3.head()

,transaction_count,drawdown_rate,fraud_rate,avg_drawdown
step,,,,
445,4,0.250000,0.000000,0.242620
483,16,0.187500,0.000000,0.127306
630,11,0.181818,0.000000,0.087587
338,23,0.173913,0.000000,0.112854
531,19,0.157895,0.052632,0.135037
